# ZenFit Meal Classifier — Colab GPU Workflow
Run cells in order. Every expensive or state-changing action is opt-in. Dataset and model state are always revalidated from disk.

## 1. Runtime verification

In [105]:
import os, sys, json, platform, subprocess, hashlib, shutil, time
from pathlib import Path
print({'python':sys.version,'platform':platform.platform(),'cwd':os.getcwd()})
IN_COLAB='google.colab' in sys.modules
print('Google Colab runtime:',IN_COLAB)

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'cwd': '/content/ZenFit/backend'}
Google Colab runtime: True


## 2. Repository setup

In [106]:
REPO_PATH=Path(os.getenv('ZENFIT_REPO_PATH','/content/ZenFit'))
REPO_URL=os.getenv('ZENFIT_REPO_URL','')
if not (REPO_PATH/'backend/training').is_dir():
    if not REPO_URL: raise FileNotFoundError('Set ZENFIT_REPO_PATH or ZENFIT_REPO_URL')
    subprocess.run(['git','clone',REPO_URL,str(REPO_PATH)],check=True)
BACKEND_PATH=REPO_PATH/'backend'; os.chdir(BACKEND_PATH)
if str(BACKEND_PATH) not in sys.path: sys.path.insert(0,str(BACKEND_PATH))
print('Repository:',REPO_PATH)
# For private Git, use a short-lived Colab secret/environment credential helper. Never embed or print a token.

Repository: /content/ZenFit


## 3. Dependency setup

In [107]:
INSTALL_DEPS=False
if INSTALL_DEPS: subprocess.run([sys.executable,'-m','pip','install','-r','requirements-training.txt'],check=True)
import torch, torchvision, numpy, pandas, sklearn, PIL
from PIL import Image
print({'torch':torch.__version__,'torchvision':torchvision.__version__,'numpy':numpy.__version__,'pandas':pandas.__version__,'sklearn':sklearn.__version__,'Pillow':PIL.__version__})

{'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.0.2', 'pandas': '2.2.2', 'sklearn': '1.6.1', 'Pillow': '11.3.0'}


## 4. GPU verification

In [108]:
print('torch.cuda.is_available():',torch.cuda.is_available()); print('CUDA version:',torch.version.cuda)
if torch.cuda.is_available():
    props=torch.cuda.get_device_properties(0); print('GPU name:',torch.cuda.get_device_name(0)); print('Total GPU memory (GiB):',round(props.total_memory/2**30,2)); print('Allocated (GiB):',round(torch.cuda.memory_allocated()/2**30,3)); print('Reserved (GiB):',round(torch.cuda.memory_reserved()/2**30,3))
else: print('CUDA is unavailable. Dataset checks may run, but training cells will fail before training.')

torch.cuda.is_available(): True
CUDA version: 12.8
GPU name: Tesla T4
Total GPU memory (GiB): 14.56
Allocated (GiB): 0.0
Reserved (GiB): 0.0


## 5. Paths and configuration

In [109]:
from collections import Counter
LOCAL_ROOT=Path(os.getenv('ZENFIT_COLAB_ROOT','/content/zenfit-work'))
RAW_ROOT=LOCAL_ROOT/'data/raw/kaggle'; DATASET=LOCAL_ROOT/'data/training/indian_food_v2'; REPORTS=LOCAL_ROOT/'reports'; MODELS=LOCAL_ROOT/'models/indian_food'; PACKAGES=LOCAL_ROOT/'artifacts'
raw=RAW_ROOT/'food_image_classification'/'Food Classification dataset'; manifest_path=DATASET/'split_manifest.json'; DRIVE_ROOT=None
for item in (RAW_ROOT,REPORTS,MODELS,PACKAGES): item.mkdir(parents=True,exist_ok=True)
IMAGE_SUFFIXES={'.jpg','.jpeg','.png','.webp','.bmp'}
def validate_prepared_dataset(dataset):
    dataset=Path(dataset); splits=('train','val','test'); errors=[]; manifest=None
    actual={s:sum(1 for p in (dataset/s).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES) if (dataset/s).is_dir() else 0 for s in splits}
    for s in splits:
        if not (dataset/s).is_dir(): errors.append(f'{s} directory is missing')
        elif actual[s]==0: errors.append(f'{s} split is empty')
    path=dataset/'split_manifest.json'
    if not path.is_file(): errors.append('split_manifest.json is missing')
    else:
        try: manifest=json.loads(path.read_text())
        except (OSError,json.JSONDecodeError) as exc: errors.append(f'split_manifest.json cannot be parsed: {exc}')
    expected={s:0 for s in splits}
    if manifest is not None:
        files=manifest.get('files')
        if not isinstance(files,list) or not files: errors.append('manifest files list is missing or empty')
        else:
            for i,row in enumerate(files):
                value=row.get('path') if isinstance(row,dict) else None; parts=Path(value).parts if isinstance(value,str) and value else ()
                if not parts or parts[0] not in splits: errors.append(f'manifest file {i} has an invalid split path')
                else: expected[parts[0]]+=1
            for s in splits:
                if expected[s]!=actual[s]: errors.append(f'{s} count mismatch: manifest={expected[s]}, actual={actual[s]}')
    return {'valid':not errors,'manifest':manifest if not errors else None,'manifest_counts':expected,'actual_counts':actual,'errors':errors}
print({'raw':str(raw),'prepared':str(DATASET),'reports':str(REPORTS),'models':str(MODELS)})

{'raw': '/content/zenfit-work/data/raw/kaggle/food_image_classification/Food Classification dataset', 'prepared': '/content/zenfit-work/data/training/indian_food_v2', 'reports': '/content/zenfit-work/reports', 'models': '/content/zenfit-work/models/indian_food'}


## 6. Kaggle authentication

In [110]:
def configure_kaggle_auth():
    token=os.getenv('KAGGLE_API_TOKEN')
    if not token and IN_COLAB:
        from google.colab import userdata
        token=userdata.get('KAGGLE_API_TOKEN')
    if token: os.environ['KAGGLE_API_TOKEN']=token
    if not (token or (Path.home()/'.kaggle/kaggle.json').is_file()): raise RuntimeError('Configure Kaggle via a Colab secret, environment variable, or secure kaggle.json')
    print('Kaggle credentials are configured (secret not displayed).')

## 7. Dataset acquisition

In [111]:
DOWNLOAD_DATASET=False
if DOWNLOAD_DATASET:
    configure_kaggle_auth(); subprocess.run([sys.executable,'training/download_kaggle_datasets.py','--dataset','food_image_classification','--root',str(RAW_ROOT)],check=True)
print('Raw dataset available:',raw.is_dir())

Raw dataset available: True


## 8. Dataset validation and preparation

In [112]:
if not raw.is_dir(): raise FileNotFoundError(f'Raw dataset is missing: {raw}. Run dataset acquisition first.')
status=validate_prepared_dataset(DATASET)
if status['valid']:
    manifest=status['manifest']; print('Prepared dataset is valid. Reusing existing dataset.')
else:
    manifest=None
    if DATASET.exists(): print('Incomplete prepared dataset detected. Regenerating.'); shutil.rmtree(DATASET)
    else: print('Prepared dataset not found. Creating train/val/test splits.')
    subprocess.run([sys.executable,'training/prepare_class_labeled_v2.py','--raw',str(raw),'--output',str(DATASET),'--reports',str(REPORTS)],check=True)
    status=validate_prepared_dataset(DATASET)
    if not status['valid']: manifest=None; raise RuntimeError('Dataset preparation finished but validation failed: '+'; '.join(status['errors']))
    manifest=json.loads(manifest_path.read_text()); print('Prepared dataset created and validated.')
print({'manifest_counts':status['manifest_counts'],'actual_counts':status['actual_counts']})

Prepared dataset is valid. Reusing existing dataset.
{'manifest_counts': {'train': 2747, 'val': 586, 'test': 596}, 'actual_counts': {'train': 2747, 'val': 586, 'test': 596}}


## 9. Split verification

In [113]:
status=validate_prepared_dataset(DATASET)
if not status['valid']: manifest=None; raise RuntimeError('Prepared dataset is invalid: '+'; '.join(status['errors']))
manifest=status['manifest']; print('Manifest counts:',status['manifest_counts']); print('Actual file counts:',status['actual_counts']); assert status['manifest_counts']==status['actual_counts']
print('Class distribution:',json.dumps({n:{k:v for k,v in row.items() if k in ('total','train','val','test')} for n,row in manifest.get('classes',{}).items()},indent=2))

Manifest counts: {'train': 2747, 'val': 586, 'test': 596}
Actual file counts: {'train': 2747, 'val': 586, 'test': 596}
Class distribution: {
  "chapati": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "chicken_curry": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "chole_bhature": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "dal_makhani": {
    "total": 276,
    "train": 193,
    "val": 41,
    "test": 42
  },
  "dhokla": {
    "total": 229,
    "train": 160,
    "val": 34,
    "test": 35
  },
  "dosa": {
    "total": 261,
    "train": 182,
    "val": 39,
    "test": 40
  },
  "fried_rice": {
    "total": 300,
    "train": 210,
    "val": 45,
    "test": 45
  },
  "idli": {
    "total": 287,
    "train": 200,
    "val": 43,
    "test": 44
  },
  "jalebi": {
    "total": 262,
    "train": 183,
    "val": 39,
    "test": 40
  },
  "kadai_paneer": {
    "total": 300,
    "train": 210,
    "val": 45,
   

## 10. Duplicate and leakage verification

In [114]:
status=validate_prepared_dataset(DATASET)
if not status['valid']: raise RuntimeError('Prepared dataset is invalid: '+'; '.join(status['errors']))
manifest=status['manifest']; missing=[i for i,row in enumerate(manifest['files']) if not row.get('sha256')]
if missing: raise RuntimeError(f'Manifest is missing sha256 for {len(missing)} files')
hash_splits={}
for row in manifest['files']: hash_splits.setdefault(row['sha256'],set()).add(Path(row['path']).parts[0])
duplicates=len(manifest['files'])-len(hash_splits); leaks={h:sorted(s) for h,s in hash_splits.items() if len(s)>1}
if duplicates: raise RuntimeError(f'Duplicate SHA256 values detected: {duplicates}')
if leaks: raise RuntimeError(f'Cross-split leakage detected: {len(leaks)} hashes')
print('No duplicate SHA256 values or cross-split leakage across',len(hash_splits),'files')

No duplicate SHA256 values or cross-split leakage across 3929 files


## 11. Model configuration

In [115]:
CONFIG=Path('training/configs/indian_food_v2_candidate.json'); cfg=json.loads(CONFIG.read_text())
VERSION=os.getenv('ZENFIT_MODEL_VERSION','1.2.0-colab-candidate'); BASELINE_VERSION=os.getenv('ZENFIT_BASELINE_VERSION','1.1.0')
SMOKE_VERSION=VERSION+'-smoke'; SMOKE_ROOT=MODELS/SMOKE_VERSION; CANDIDATE=MODELS/VERSION
print(json.dumps(cfg,indent=2)); print({'smoke':str(SMOKE_ROOT),'candidate':str(CANDIDATE)})

{
  "architecture": "efficientnet_b0",
  "image_size": 224,
  "batch_size": 32,
  "epochs": 12,
  "learning_rate": 0.0003,
  "head_epochs": 4,
  "finetune_epochs": 8,
  "head_learning_rate": 0.001,
  "finetune_learning_rate": 8e-05,
  "unfreeze_last_blocks": 3,
  "early_stopping_patience": 4,
  "weighted_loss": true,
  "random_seed": 42,
  "augmentation": {
    "random_resized_crop_scale": [
      0.8,
      1.0
    ],
    "horizontal_flip_probability": 0.5,
    "rotation_degrees": 8,
    "brightness": 0.15,
    "contrast": 0.15,
    "saturation": 0.1
  }
}
{'smoke': '/content/zenfit-work/models/indian_food/1.2.0-colab-candidate-smoke', 'candidate': '/content/zenfit-work/models/indian_food/1.2.0-colab-candidate'}


## 12. Trained candidate and optional Drive backup

In [ ]:
VERSION='1.2.0-colab-candidate'; CANDIDATE=MODELS/VERSION; ARTIFACT=PACKAGES/VERSION
OPEN_SET_MANIFEST=REPORTS/f'{VERSION}-open-set-manifest.json'; OPEN_SET_PREDICTIONS=REPORTS/f'{VERSION}-open-set-predictions.json'; OPEN_SET_REPORT=REPORTS/f'{VERSION}-open-set-evaluation.json'; THRESHOLD_REPORT=REPORTS/f'{VERSION}-threshold-report.json'; LATENCY_REPORT=REPORTS/f'{VERSION}-latency.json'; RELEASE_EVIDENCE=CANDIDATE/'release_evidence.json'; INFERENCE_REPORT=REPORTS/f'{VERSION}-independent-inference-smoke.json'
MOUNT_DRIVE=False; BACKUP_CANDIDATE_NOW=False; DRIVE_ROOT=Path('/content/drive/MyDrive/ZenFit')
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if BACKUP_CANDIDATE_NOW:
    if not Path('/content/drive').is_mount(): raise RuntimeError('Google Drive must already be mounted')
    required=[CANDIDATE/name for name in ('model.pt','metrics.json','calibration.json')]; missing=[str(x) for x in required if not x.is_file()]
    if missing: raise RuntimeError('Candidate backup blocked; missing: '+', '.join(missing))
    destination=DRIVE_ROOT/'backups'/VERSION/'candidate-model'
    if destination.exists(): raise FileExistsError(f'Refusing to overwrite {destination}')
    shutil.copytree(CANDIDATE,destination); total=sum(x.stat().st_size for x in destination.rglob('*') if x.is_file())
    from training.open_set_workflow import sha256
    print({'destination':str(destination),'total_bytes':total,'model_pt_sha256':sha256(destination/'model.pt')})
print('Immutable candidate:',CANDIDATE)

## 13. Open-set evidence acquisition and manifest

In [ ]:
GENERATE_OPEN_SET_EVIDENCE=False; ACQUIRE_RESEARCH_NON_FOOD=False
NON_FOOD_ROOT=LOCAL_ROOT/'open_set/non_food'; NON_FOOD_ROOT.mkdir(parents=True,exist_ok=True)
if ACQUIRE_RESEARCH_NON_FOOD:
    from torchvision.datasets import Caltech101
    source=Caltech101(root=str(LOCAL_ROOT/'downloads'),download=True); categories={'Faces','Motorbikes','airplanes','car_side','chair','laptop','watch','camera','cellphone'}; written=0
    for index,(image,target) in enumerate(source):
        category=source.categories[target]
        if category in categories:
            target_path=NON_FOOD_ROOT/category/f'{index}.png'; target_path.parent.mkdir(parents=True,exist_ok=True); image.convert('RGB').save(target_path); written+=1
            if written>=150: break
    (NON_FOOD_ROOT/'source_manifest.json').write_text(json.dumps({'source':'Caltech101 via torchvision','license':'UNVERIFIED','license_review_status':'pending','research_only':True,'notes':'Developer-beta research evidence only; not eligible for production license gate.'},indent=2))
if GENERATE_OPEN_SET_EVIDENCE:
    from training.open_set_workflow import build_evidence_manifest
    result=build_evidence_manifest(prepared_dataset=DATASET,raw_food_root=raw,non_food_root=NON_FOOD_ROOT,output=OPEN_SET_MANIFEST,per_group=100)
    if not result['valid']: raise RuntimeError('; '.join(result['errors']))
    print({'manifest':str(OPEN_SET_MANIFEST),'counts':result['counts']})
else: print('Evidence generation disabled. Provide licensed non-food images plus source_manifest.json, or explicitly acquire research-only evidence.')

## 14. Open-set prediction generation

In [ ]:
GENERATE_OPEN_SET_PREDICTIONS=False
if GENERATE_OPEN_SET_PREDICTIONS:
    from training.open_set_workflow import generate_predictions,validate_evidence_manifest
    validation=validate_evidence_manifest(OPEN_SET_MANIFEST,DATASET/'split_manifest.json')
    if not validation['valid']: raise RuntimeError('; '.join(validation['errors']))
    payload=generate_predictions(candidate=CANDIDATE,evidence_manifest=OPEN_SET_MANIFEST,output=OPEN_SET_PREDICTIONS,device='cuda'); print({'output':str(OPEN_SET_PREDICTIONS),'rows':len(payload['predictions'])})
else: print('Open-set prediction generation disabled.')

## 15. Open-set evaluation

In [ ]:
RUN_OPEN_SET_EVALUATION=False
STARTING_THRESHOLDS=Path('training/configs/open_set_1.1.0.json')
if RUN_OPEN_SET_EVALUATION:
    if not OPEN_SET_PREDICTIONS.is_file(): raise FileNotFoundError('Generate open-set predictions first')
    from training.open_set_workflow import enrich_open_set_evaluation
    print(enrich_open_set_evaluation(OPEN_SET_PREDICTIONS,STARTING_THRESHOLDS,OPEN_SET_REPORT))
else: print('Open-set evaluation disabled.')

## 16. Threshold search

In [ ]:
RUN_THRESHOLD_SEARCH=False
if RUN_THRESHOLD_SEARCH:
    if not OPEN_SET_PREDICTIONS.is_file(): raise FileNotFoundError('Generate open-set predictions first')
    payload=json.loads(OPEN_SET_PREDICTIONS.read_text()); rows=payload['predictions']
    from training.open_set_evaluation import threshold_sweep
    from training.analyze_open_set_thresholds import recommend
    sweep=threshold_sweep(rows,VERSION,confidence_values=(.40,.45,.50,.55,.57,.60,.65,.70,.75,.80),margin_values=(.03,.05,.08,.10,.12,.15,.20,.25),entropy_values=(None,.8,1.0,1.2,1.5,1.8,2.0)); selected=recommend(sweep); selected['status']='DEVELOPER_BETA'; selected['sweep']=sweep
    THRESHOLD_REPORT.write_text(json.dumps(selected,indent=2)); (CANDIDATE/'open_set_thresholds.json').write_text(json.dumps(selected['thresholds'],indent=2))
    from training.open_set_workflow import enrich_open_set_evaluation
    enrich_open_set_evaluation(OPEN_SET_PREDICTIONS,CANDIDATE/'open_set_thresholds.json',OPEN_SET_REPORT)
    table=[]
    for row in sweep:
        t=row['thresholds']; m=row['metrics']; table.append({'confidence':t['supported_food_min_confidence'],'margin':t['min_top1_top2_margin'],'entropy':t['max_entropy'],'known_acceptance':1-m['supported_food']['false_rejection_rate'],'known_false_rejection':m['supported_food']['false_rejection_rate'],'unknown_rejection':m['unknown_food']['rejection_rate'],'non_food_rejection':m['non_food']['rejection_rate']})
    display(pandas.DataFrame(table).sort_values(['unknown_rejection','non_food_rejection','known_acceptance'],ascending=False)); print('Developer-beta recommendation:',selected['thresholds'])
else: print('Threshold search disabled.')

## 17. Latency benchmark

In [ ]:
RUN_LATENCY_BENCHMARK=False
if RUN_LATENCY_BENCHMARK:
    known=sorted(path for path in (DATASET/'test').rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
    from training.open_set_workflow import benchmark_latency
    print(benchmark_latency(candidate=CANDIDATE,known_images=known,output=LATENCY_REPORT,runs=50,warmups=10))
else: print('Latency benchmark disabled.')
# Training GPU peak memory is recorded inside train_indian_food.py as peak_cuda_memory_bytes. Parent-process CUDA counters are intentionally not reported.

## 18. Regression evidence

In [ ]:
historical={'comparison_type':'historical_metadata_comparison','direct_same_runtime_binary_comparison':False,'production_regression_gate':'BLOCKED','baseline':{'version':'1.1.0','accuracy':.8943,'macro_f1':.8974,'top_3_accuracy':.9849},'candidate':{'version':VERSION,'accuracy':.8959731543624161,'macro_f1':.898781103967839,'top_3_accuracy':.9832214765100671}}
print(json.dumps(historical,indent=2))

## 19. Release evidence and model card

In [ ]:
GENERATE_RELEASE_EVIDENCE=False
if GENERATE_RELEASE_EVIDENCE:
    required=[OPEN_SET_REPORT,THRESHOLD_REPORT,LATENCY_REPORT]; missing=[str(path) for path in required if not path.is_file()]
    if missing: raise RuntimeError('Release evidence blocked; missing: '+', '.join(missing))
    from training.open_set_workflow import generate_release_evidence,update_model_card
    release=generate_release_evidence(candidate=CANDIDATE,open_set_report=OPEN_SET_REPORT,threshold_report=THRESHOLD_REPORT,latency_report=LATENCY_REPORT,output=RELEASE_EVIDENCE); update_model_card(CANDIDATE,release); print(json.dumps(release,indent=2))
else: print('Release-evidence generation disabled.')

## 20. Developer-beta and strict-production readiness

In [ ]:
if not RELEASE_EVIDENCE.is_file(): print('Readiness unavailable: generate release evidence first.')
else:
    release=json.loads(RELEASE_EVIDENCE.read_text()); print('Developer beta:',release['developer_beta']['status']); print(json.dumps(release['developer_beta']['checks'],indent=2)); print('PRODUCTION_APPROVED =',release['production']['approved']); print('Production reason:',release['production']['reason'])

## 21. Developer-beta artifact export

In [ ]:
EXPORT_ARTIFACT=False; WRITE_DEVELOPER_BETA_POINTER=False
if EXPORT_ARTIFACT:
    if not RELEASE_EVIDENCE.is_file(): raise FileNotFoundError('release_evidence.json is required')
    release=json.loads(RELEASE_EVIDENCE.read_text())
    if release['developer_beta']['status']!='DEVELOPER_BETA_READY': raise RuntimeError('Developer-beta readiness is blocked')
    required=('model.pt','classes.json','config.json','metrics.json','calibration.json','dataset_manifest.json','open_set_thresholds.json','release_evidence.json','model_card.md'); missing=[name for name in required if not (CANDIDATE/name).is_file()]
    if missing: raise RuntimeError('Artifact export blocked; missing: '+', '.join(missing))
    if ARTIFACT.exists(): raise FileExistsError(f'Refusing to overwrite {ARTIFACT}')
    subprocess.run([sys.executable,'scripts/package_model_artifact.py',str(CANDIDATE),str(ARTIFACT),'--environment','developer-beta'],check=True)
    from app.ai.artifacts import verify_artifact
    verified=verify_artifact(ARTIFACT,required_environment='developer-beta'); print({'artifact':str(ARTIFACT),'manifest':verified})
    if WRITE_DEVELOPER_BETA_POINTER:
        pointer=MODELS/'developer_beta.json'
        if pointer.exists(): raise FileExistsError(f'Refusing to overwrite {pointer}')
        pointer.write_text(json.dumps({'version':VERSION,'status':'DEVELOPER_BETA','artifact_verified':True,'manual_correction_required':True,'confidence_required':True,'top_k_required':True},indent=2)); print('Wrote',pointer)
else: print('Artifact export disabled. active.json is never written.')

## 22. Independent packaged inference smoke

In [ ]:
RUN_INFERENCE_SMOKE=False
if RUN_INFERENCE_SMOKE:
    from app.ai.artifacts import verify_artifact
    from app.ai.meal_scan.open_set import Candidate,OpenSetDecisionEngine,OpenSetInput,OpenSetThresholds,probability_entropy
    from training.open_set_workflow import load_candidate
    verify_artifact(ARTIFACT,required_environment='developer-beta'); model,labels,config,calibration,transform=load_candidate(ARTIFACT,'cuda'); thresholds=OpenSetThresholds.from_json(ARTIFACT/'open_set_thresholds.json')
    manifest=json.loads(OPEN_SET_MANIFEST.read_text()); groups={truth:[Path(item['path']) for item in manifest['items'] if item['truth']==truth][:5] for truth in ('supported_food','unknown_food','non_food')}
    if any(not paths for paths in groups.values()): raise RuntimeError('All three evidence groups need samples')
    results=[]
    from app.ai.meal_scan.open_set import probability_entropy
    for truth,paths in groups.items():
        for path in paths:
            image=Image.open(path).convert('RGB'); torch.cuda.synchronize(); started=time.perf_counter()
            with torch.inference_mode(): probs=(model(transform(image).unsqueeze(0).cuda())/calibration['temperature']).softmax(1)[0].cpu()
            torch.cuda.synchronize(); values,indices=probs.topk(min(3,len(labels))); top=tuple(Candidate(labels[int(i)],float(v)) for v,i in zip(values,indices)); decision=OpenSetDecisionEngine(thresholds).decide(OpenSetInput(top_candidates=top,entropy=probability_entropy(probs),model_version=VERSION)); row={'truth_group':truth,'filename':path.name,'predicted_class':top[0].label,'confidence':top[0].confidence,'top_3':[{'label':x.label,'confidence':x.confidence} for x in top],'decision':decision.decision.value,'latency_ms':(time.perf_counter()-started)*1000}; results.append(row); print(row)
    INFERENCE_REPORT.write_text(json.dumps(results,indent=2)); del model; torch.cuda.empty_cache()
else: print('Independent packaged inference smoke disabled.')

## 23. Optional final Drive backup

In [ ]:
BACKUP_AFTER_EXPORT=False
if BACKUP_AFTER_EXPORT:
    if not Path('/content/drive').is_mount(): raise RuntimeError('Google Drive must already be mounted')
    destination=DRIVE_ROOT/'backups'/VERSION/'final-evidence-and-artifact'
    if destination.exists(): raise FileExistsError(f'Refusing to overwrite {destination}')
    destination.mkdir(parents=True); sources=[RELEASE_EVIDENCE,THRESHOLD_REPORT,OPEN_SET_MANIFEST,OPEN_SET_PREDICTIONS,OPEN_SET_REPORT,LATENCY_REPORT,INFERENCE_REPORT]
    for source in sources:
        if source.is_file(): shutil.copy2(source,destination/source.name)
    if ARTIFACT.is_dir(): shutil.copytree(ARTIFACT,destination/'artifact')
    print('Final backup:',destination)
else: print('Final backup disabled. Secrets are never copied to Drive.')

## 24. Dynamic final summary

In [ ]:
state={'candidate_trained':(CANDIDATE/'metrics.json').is_file(),'open_set_manifest':OPEN_SET_MANIFEST.is_file(),'predictions':OPEN_SET_PREDICTIONS.is_file(),'open_set_evaluation':OPEN_SET_REPORT.is_file(),'thresholds':(CANDIDATE/'open_set_thresholds.json').is_file() and THRESHOLD_REPORT.is_file(),'latency':LATENCY_REPORT.is_file(),'release_evidence':RELEASE_EVIDENCE.is_file(),'artifact':(ARTIFACT/'artifact_manifest.json').is_file(),'independent_smoke':INFERENCE_REPORT.is_file()}
if not state['candidate_trained']: next_step='Candidate model is missing; restore the trained candidate backup.'
elif not state['open_set_manifest']: next_step='Generate open-set evidence.'
elif not state['predictions']: next_step='Generate open-set predictions.'
elif not state['open_set_evaluation']: next_step='Run open-set evaluation.'
elif not state['thresholds']: next_step='Run threshold search.'
elif not state['latency']: next_step='Run latency benchmark.'
elif not state['release_evidence']: next_step='Generate release evidence.'
elif not state['artifact']: next_step='Export developer-beta artifact.'
elif not state['independent_smoke']: next_step='Run packaged artifact inference smoke.'
else: next_step='Developer-beta artifact is ready for deployment planning.'
print(state); print('Next step:',next_step); print('PRODUCTION_APPROVED = False')